# NB01 — Identity-Disjoint Specimen Construction and Splits (run first)

Builds a grouping key, `specimen_id`, that identifies which images in BDLitchi depict
the same physical leaf, and produces the train/validation/test partitions used
throughout the rest of this project. Perceptual-hash (pHash) clustering alone catches
burst-shot near-duplicate frames but not a second photograph of the same leaf taken
from a different angle or under different lighting. `specimen_id` is defined as the
connected components of a graph in which two images are linked whenever either their
pHash Hamming distance is small, or they were captured in the same session within a
short time window. Every split downstream partitions on `specimen_id`, never on
individual images.

### Outputs (publish as a Kaggle Dataset; attached by NB02–NB06)
| file | purpose |
|---|---|
| `manifest_full.csv` | one row per image: label, session_id, cluster_id, specimen_id, split columns |
| `splits.json` | frozen split definition and counts |
| `split_diagnostics.csv` | per-class specimen/session counts, single-session flags |

### Split design
1. `common_test` — a specimen-disjoint held-out set (~15% of specimens per class),
   reserved and never trained on by any condition. This is the shared evaluation set
   used across every downstream comparison.
2. From the remaining pool:
   - **naive** condition: image-level random train/val/test (specimens can span partitions)
   - **group** condition: specimen-disjoint train/val/test
3. NB02 trains one model per condition and reports both the reporting inflation (each
   condition scored on its own test set) and the true generalisation gap (both
   conditions scored on `common_test`).

Runtime: ~20–40 minutes (CPU-bound: EXIF and pHash extraction over ~11k images). No GPU required.


# NB01b — Hamming-threshold sensitivity of the specimen construction (R7 point 9)

Reviewer 7 on the IPA submission asked whether the leakage-ablation inflation numbers
(+0.38pp ResNet-50 / +4.04pp ViT-Base/16 at the paper's chosen Hamming threshold of 5)
are robust to that threshold choice, not just whether the *specimen count* is robust
(`hamming_sensitivity.csv` / manuscript Table 4 already covers that, but not the
downstream ablation result itself).

This notebook reuses NB01's EXIF-scan and pHash-code cells UNCHANGED (they do not
depend on the threshold), then rebuilds specimens/splits at two extra thresholds
bracketing the paper's choice of 5: **4** (stricter) and **6** (looser). It writes a
separate `manifest_full.csv` per threshold, exactly like NB01's normal output, so
NB02b (the companion notebook) can run the leakage ablation on each one unchanged.

**Compute note:** rebuilding specimens/splits here is cheap (CPU, seconds). The
expensive part is NB02b's GPU training, which this notebook does NOT do. Run this
notebook first, upload its output as a Kaggle dataset, then run NB02b against it.

Edit `HAMMING_THRESHOLDS` below if you want more/fewer sweep points — each extra
threshold costs NB02b roughly one more full ablation run (2 archs x N seeds x 2
conditions).

In [1]:
# ===== Imports =====
import os, json, time, random, warnings, math
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import timm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, matthews_corrcoef, confusion_matrix, roc_auc_score)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


device: cuda | Tesla T4


In [2]:
# ===== Config =====
RAW_DATASET_DIR = '/kaggle/input/datasets/alenmaruf/bdlithi/Dataset'   # <-- EDIT: folder containing the 11 class sub-folders
OUT_DIR = Path('/kaggle/working/revision_splits_hamming_sweep'); OUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED          = 42
SESSION_GAP_SECONDS  = 300    # unchanged from NB01 - does not depend on the pHash threshold
SPECIMEN_BLOCK_SECONDS = 30   # unchanged from NB01 - the paper's chosen value (Table 5)

# The paper's main-text threshold is 5 (already have manifest_full.csv for this from
# the original NB01 run - no need to regenerate it here). 3 and 4 are stricter extra
# points (more specimens, always viable). 6 is kept here DELIBERATELY even though it is
# expected to fail the same 20-specimens-per-class floor hamming_sensitivity.csv already
# documents for threshold >= 6 - the sweep cell below catches that and records it as an
# "infeasible" result rather than crashing, which is itself evidence for the robustness
# argument (there is no viable looser threshold to compare against).
HAMMING_THRESHOLDS = [3]   # 4 already succeeded, 6 already confirmed infeasible - only 3 is new work

COMMON_TEST_FRAC     = 0.15
VAL_FRAC             = 0.15
TEST_FRAC            = 0.15

print('raw dir exists:', Path(RAW_DATASET_DIR).exists())
print('Will build manifests for Hamming thresholds:', HAMMING_THRESHOLDS)


raw dir exists: True
Will build manifests for Hamming thresholds: [3]


In [3]:
def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(MASTER_SEED)

In [4]:
# ===== 1. Scan images + read EXIF timestamps =====
import imagehash
from PIL import ExifTags

EXIF_DT = {v:k for k,v in ExifTags.TAGS.items()}.get('DateTimeOriginal')

def exif_ts(p):
    try:
        ex = Image.open(p)._getexif()
        if ex and EXIF_DT in ex:
            return time.mktime(time.strptime(ex[EXIF_DT], '%Y:%m:%d %H:%M:%S'))
    except Exception:
        pass
    return None

rows=[]
classes = sorted([d.name for d in Path(RAW_DATASET_DIR).iterdir() if d.is_dir()])
print('classes:', len(classes), classes)
for c in classes:
    for p in sorted((Path(RAW_DATASET_DIR)/c).glob('*')):
        if p.suffix.lower() not in ('.jpg','.jpeg','.png','.bmp','.webp'): continue
        rows.append({'filepath':str(p),'filename':p.name,'label':c,
                     'ts':exif_ts(p),'mtime':p.stat().st_mtime})
df=pd.DataFrame(rows)
print('images found:', len(df), '| with EXIF timestamp:', df.ts.notna().sum())

# fall back to file mtime where EXIF is missing, so session logic still works
df['ts_eff'] = df.ts.fillna(df.mtime)
df = df.sort_values(['label','ts_eff']).reset_index(drop=True)
df.head()

classes: 11 ['Black Spot', 'Burned Leaf', 'Dried Leaf', 'Fungal Stripe Damage', 'Healthy Leaf', 'Insect Chewing Damage', 'Leaf Blight Disease', 'Pest-Affected Dry Leaf', 'Red Rust Disease', 'White Spot', 'Yellow Mosaic Virus']
images found: 11094 | with EXIF timestamp: 11094


,filepath,filename,label,ts,mtime,ts_eff
0,/kaggle/input/datasets/alenmaruf/bdlithi/Datas...,IMG20250601101600.jpg,Black Spot,1.748773e+09,1.794316e+09,1.748773e+09
1,/kaggle/input/datasets/alenmaruf/bdlithi/Datas...,IMG20250601101612.jpg,Black Spot,1.748773e+09,1.794316e+09,1.748773e+09
2,/kaggle/input/datasets/alenmaruf/bdlithi/Datas...,IMG20250601101626.jpg,Black Spot,1.748773e+09,1.794316e+09,1.748773e+09
3,/kaggle/input/datasets/alenmaruf/bdlithi/Datas...,IMG20250601101635.jpg,Black Spot,1.748773e+09,1.794316e+09,1.748773e+09
4,/kaggle/input/datasets/alenmaruf/bdlithi/Datas...,IMG20250601101759.jpg,Black Spot,1.748773e+09,1.794316e+09,1.748773e+09


In [5]:
# ===== 2. Sessions (time-gap based, within class) =====
sess=[]; sid=0; prev_lab=None; prev_t=None
for r in df.itertuples():
    if prev_lab!=r.label or prev_t is None or (r.ts_eff-prev_t)>SESSION_GAP_SECONDS:
        sid+=1
    sess.append(f'sess{sid:04d}'); prev_lab, prev_t = r.label, r.ts_eff
df['session_id']=sess
print('sessions:', df.session_id.nunique())
print(df.groupby('label').session_id.nunique().rename('n_sessions'))

sessions: 26
label
Black Spot                4
Burned Leaf               3
Dried Leaf                2
Fungal Stripe Damage      1
Healthy Leaf              3
Insect Chewing Damage     4
Leaf Blight Disease       1
Pest-Affected Dry Leaf    2
Red Rust Disease          4
White Spot                1
Yellow Mosaic Virus       1
Name: n_sessions, dtype: int64


In [6]:
# ===== 3. pHash =====
t0=time.time()
df['phash']=[imagehash.phash(Image.open(p)) for p in df.filepath]
print(f'pHash done in {time.time()-t0:.0f}s')

pHash done in 56s


In [7]:
# ===== Build specimens + splits for each extra Hamming threshold =====
# phash_edges below depends on PHASH_THRESHOLD; everything upstream (df, EXIF scan,
# session_id, raw phash codes in `df.phash`) was already computed in the cells above,
# unchanged from NB01, and is threshold-independent so it is NOT recomputed here.

class UF:
    def __init__(s,n): s.p=list(range(n))
    def find(s,x):
        while s.p[x]!=x: s.p[x]=s.p[s.p[x]]; x=s.p[x]
        return x
    def union(s,a,b):
        ra,rb=s.find(a),s.find(b)
        if ra!=rb: s.p[rb]=ra

def popcount64(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0f0f0f0f0f0f0f0f)
    return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)

hbits = np.array([int(str(h),16) for h in df.phash], dtype=np.uint64)

def build_specimens_at_threshold(threshold, block_s=SPECIMEN_BLOCK_SECONDS):
    t0=time.time()
    phash_edges=[]
    for lab, g in df.groupby('label'):
        idx=g.index.to_numpy(); H=hbits[idx]
        D=popcount64(H[:,None]^H[None,:])
        iu,ju=np.triu_indices(len(idx),k=1)
        near=D[iu,ju]<=threshold
        phash_edges.extend(zip(idx[iu[near]], idx[ju[near]]))
    uf=UF(len(df))
    for i_,j_ in phash_edges: uf.union(int(i_),int(j_))
    if block_s>0:
        blk=np.empty(len(df),dtype=np.int64); b=0
        for _,g in df.groupby('session_id', sort=False):
            gi=g.index.to_numpy(); gt=g.ts_eff.values
            o=np.argsort(gt,kind='stable'); gi,gt=gi[o],gt[o]
            b+=1; start=gt[0]
            for k in range(len(gi)):
                if gt[k]-start > block_s:
                    b+=1; start=gt[k]
                blk[gi[k]]=b
        o=np.argsort(blk,kind='stable'); bs=blk[o]
        starts=np.flatnonzero(np.r_[True, bs[1:]!=bs[:-1]])
        for s_,e_ in zip(starts, np.r_[starts[1:], len(o)]):
            grp=o[s_:e_]
            for x in grp[1:]: uf.union(int(grp[0]), int(x))
    sid = np.array([uf.find(i_) for i_ in range(len(df))])
    print(f'  threshold={threshold}: {len(phash_edges)} pHash edges, '
          f'{len(np.unique(sid))} specimens ({time.time()-t0:.0f}s)')
    return sid

def _stable_seed(base, key):
    import zlib
    return (base + zlib.crc32(str(key).encode())) % (2**31-1)

def specimen_split(frame, fracs, seed, min_per_part=1, verbose=False):
    # identical to NB01's specimen_split - copied verbatim, see NB01 for full comments.
    names=list(fracs); nparts=len(names)
    out=pd.Series('__none__', index=frame.index)
    starved=[]
    for lab,g in frame.groupby('label'):
        sizes=g.groupby('specimen_id').size()
        specs=list(sizes.index)
        r=np.random.RandomState(_stable_seed(seed,lab)); r.shuffle(specs)
        specs=sorted(specs, key=lambda s: (-int(sizes[s]), str(s)))
        n=len(specs)
        if n < nparts*min_per_part:
            big=max(fracs,key=fracs.get)
            out[g.index]=big; starved.append((lab,n)); continue
        total=int(sizes.sum())
        target={k: fracs[k]*total for k in names}
        cur={k:0 for k in names}; assign={}
        for s in specs:
            k=max(names, key=lambda k:(target[k]-cur[k], k))
            assign[s]=k; cur[k]+=int(sizes[s])
        for _ in range(nparts*3):
            counts={k:sum(1 for v in assign.values() if v==k) for k in names}
            short=[k for k in names if counts[k]<min_per_part]
            if not short: break
            k_short=short[0]
            donors=[s for s,v in assign.items() if counts[v]>min_per_part]
            if not donors: break
            s_move=min(donors, key=lambda s:int(sizes[s]))
            cur[assign[s_move]]-=int(sizes[s_move]); assign[s_move]=k_short
            cur[k_short]+=int(sizes[s_move])
        out[g.index]=g.specimen_id.map(assign)
    if starved:
        print(f'  WARNING: {len(starved)} class(es) starved: {starved}')
    assert (out!='__none__').all(), 'some rows were never assigned'
    return out

infeasible_thresholds = []
for T in HAMMING_THRESHOLDS:
    print(f'\n{"="*60}\nHAMMING THRESHOLD = {T}\n{"="*60}')
    try:
        tdf = df.copy()
        sid = build_specimens_at_threshold(T)
        tdf['specimen_id'] = [f's{v:06d}' for v in sid]

        ps = tdf.groupby('label').specimen_id.nunique()
        print('  specimens per class (min = %d, class = %s):' % (ps.min(), ps.idxmin()))
        print(ps.to_string())
        if ps.min() < 20:
            print(f'  *** WARNING: smallest class has only {ps.min()} specimens at threshold={T}.')
            print(f'      This is below the 20-specimen floor NB01 requires - continuing to attempt')
            print(f'      the split anyway, since failure here is itself a reportable result. ***')

        rng = np.random.RandomState(MASTER_SEED)
        tdf['reserve'] = specimen_split(tdf, {'common_test':COMMON_TEST_FRAC,'pool':1-COMMON_TEST_FRAC}, MASTER_SEED)
        common_test = tdf[tdf.reserve=='common_test'].copy()
        pool        = tdf[tdf.reserve=='pool'].copy()

        g_assign = specimen_split(pool, {'train':1-VAL_FRAC-TEST_FRAC,'val':VAL_FRAC,'test':TEST_FRAC}, MASTER_SEED+1)
        tdf.loc[pool.index,'group_split'] = g_assign

        idx=pool.index.to_numpy().copy(); rng.shuffle(idx)
        n=len(idx); nv=int(VAL_FRAC*n); nt=int(TEST_FRAC*n)
        naive=pd.Series(index=pool.index, dtype=object)
        naive[idx[:nv]]='val'; naive[idx[nv:nv+nt]]='test'; naive[idx[nv+nt:]]='train'
        tdf.loc[pool.index,'naive_split']=naive
        tdf['group_split']=tdf['group_split'].fillna('common_test')
        tdf['naive_split']=tdf['naive_split'].fillna('common_test')

        # sanity gate - same checks as NB01 cell 7, abbreviated. A failure here means this
        # threshold cannot support a clean train/val/test split and is caught below, not fatal.
        ct_spec=set(common_test.specimen_id); pool_spec=set(pool.specimen_id)
        assert len(ct_spec & pool_spec)==0, f'threshold={T}: common_test overlaps pool!'
        sp=tdf[tdf.group_split!='common_test'].groupby('specimen_id').group_split.nunique()
        assert (sp==1).all(), f'threshold={T}: a specimen spans >1 group-split part!'
        cov=tdf[tdf.group_split!='common_test'].groupby(['label','group_split']).size().unstack(fill_value=0)
        for part in ['train','val','test']:
            if part not in cov.columns: cov[part]=0
            missing=[l for l in cov.index if cov.loc[l,part]==0]
            assert not missing, f'threshold={T}: class(es) {missing} missing from {part}!'
        print(f'  OK - threshold={T} passes the same gate NB01 uses before NB02.')

        tdir = OUT_DIR / f'hamming_{T}'; tdir.mkdir(exist_ok=True)
        keep=['filepath','filename','label','ts_eff','session_id','cluster_id','specimen_id',
              'reserve','group_split','naive_split']
        tdf[[k for k in keep if k in tdf.columns]].to_csv(tdir/'manifest_full.csv', index=False)
        json.dump({'hamming_threshold':T,'specimens':int(tdf.specimen_id.nunique()),
                    'specimens_per_class':{k:int(v) for k,v in ps.items()}},
                   open(tdir/'splits_meta.json','w'), indent=2)
        print(f'  Saved {tdir}/manifest_full.csv')

    except AssertionError as e:
        print(f'  INFEASIBLE at threshold={T}: {e}')
        print(f'  No manifest written for threshold={T}. This is an expected, reportable outcome')
        print(f'  (matches the under-floor finding in hamming_sensitivity.csv for Hamming >= 6),')
        print(f'  not a bug - it means there is no viable looser threshold to compare against 5.')
        infeasible_thresholds.append(T)

print(f'\nDone. Infeasible thresholds (skipped, no manifest): {infeasible_thresholds}')
print('Viable thresholds (manifests saved):', [t for t in HAMMING_THRESHOLDS if t not in infeasible_thresholds])
print('Upload', OUT_DIR, 'as a Kaggle dataset, e.g.')
print('  "bdlitchi-hamming-sweep-splits", then point NB02b at the VIABLE thresholds only.')
print('Remember: the threshold=5 manifest already exists from the ORIGINAL NB01 run -')
print('reuse that one directly for the threshold=5 point in the comparison, do not regenerate it.')



HAMMING THRESHOLD = 3
  threshold=3: 17788 pHash edges, 1127 specimens (0s)
  specimens per class (min = 37, class = Leaf Blight Disease):
label
Black Spot                199
Burned Leaf                91
Dried Leaf                125
Fungal Stripe Damage       59
Healthy Leaf               53
Insect Chewing Damage      65
Leaf Blight Disease        37
Pest-Affected Dry Leaf    140
Red Rust Disease           51
White Spot                117
Yellow Mosaic Virus       190
  OK - threshold=3 passes the same gate NB01 uses before NB02.
  Saved /kaggle/working/revision_splits_hamming_sweep/hamming_3/manifest_full.csv

Done. Infeasible thresholds (skipped, no manifest): []
Viable thresholds (manifests saved): [3]
Upload /kaggle/working/revision_splits_hamming_sweep as a Kaggle dataset, e.g.
  "bdlitchi-hamming-sweep-splits", then point NB02b at the VIABLE thresholds only.
Remember: the threshold=5 manifest already exists from the ORIGINAL NB01 run -
reuse that one directly for the threshold